# 13-2. KAPE 결과 구조와 증거 목록 예제

## Goal

- manifest 선언과 실제 파일 상태를 대조합니다.
- 입력 CSV 해시와 수집 원본 해시를 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

임시 디렉터리에 합성 CSV와 manifest를 만듭니다.


## Steps

### 입력 범위와 누락 상태 확인

manifest에 선언한 각 파일을 processed·missing으로 분류합니다.


In [1]:
from hashlib import sha256
import json
from pathlib import Path
from tempfile import TemporaryDirectory


def inspect_manifest(manifest_path: Path):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    root = manifest_path.parent.resolve()
    coverage = []
    for source in manifest["sources"]:
        candidate = (root / source["path"]).resolve()
        if root not in candidate.parents:
            raise ValueError("입력 루트 밖의 경로입니다")
        if not candidate.exists():
            coverage.append({"id": source["id"], "status": "missing"})
            continue
        data = candidate.read_bytes()
        coverage.append({"id": source["id"], "status": "processed", "sha256": sha256(data).hexdigest()})
    return coverage


with TemporaryDirectory() as directory:
    root = Path(directory)
    (root / "events.csv").write_text("Time,EventId\n2026-09-01T00:00:00Z,4624\n", encoding="utf-8")
    manifest = {"case_id": "SYNTHETIC", "sources": [
        {"id": "events", "path": "events.csv"},
        {"id": "amcache", "path": "amcache.csv"},
    ]}
    manifest_path = root / "manifest.json"
    manifest_path.write_text(json.dumps(manifest), encoding="utf-8")
    coverage = inspect_manifest(manifest_path)
print(coverage)


[{'id': 'events', 'status': 'processed', 'sha256': '5c3b195b50073d695c7f10668bcb279ea1f7ef5fec171d2d28033b067a88c643'}, {'id': 'amcache', 'status': 'missing'}]


## Checks

처리됨과 누락을 0건과 혼동하지 않는지 확인합니다.


In [2]:
assert [item["status"] for item in coverage] == ["processed", "missing"]
assert len(coverage[0]["sha256"]) == 64
assert "sha256" not in coverage[1]
print("증거 목록 검사 통과")


증거 목록 검사 통과


## Next Steps

입력 CSV 해시는 파서 출력의 동일성을 확인할 뿐 원본 EVTX의 인계 기록을 대신하지 않습니다.
